# 🏢 SBA RAG System

SBA RAG System with AI Agent - Small Business Administration Q&A Assistant
===========================================================================
A Retrieval-Augmented Generation (RAG) system enhanced with AI agent
capabilities that provides accurate, verified answers about SBA loans,
certifications, and business resources by retrieving information from
official SBA documentation and performing financial calculations.

Key Features:
- Document retrieval from 12 authoritative SBA sources
- Semantic search with ChromaDB vector storage
- GPT-3.5-turbo powered Q&A with grounded responses
- Off-topic detection with honest "I don't know" responses
- Source attribution for transparency and verification

AI Agent Enhancement:
- Multi-tool reasoning and orchestration
- Loan payment calculator (precise amortization formula)
- Autonomous tool selection based on question type
- Multi-step workflow execution (calculate → search → synthesize)
- Safety controls (max_iterations=3 prevents infinite loops)
- Transparent tool usage tracking


---


---

##  Install Packages

In [1]:
print("📦 Installing packages...\n")

# Install with output hidden
import sys
!{sys.executable} -m pip install -q langchain==0.1.20
!{sys.executable} -m pip install -q langchain-community==0.0.38
!{sys.executable} -m pip install -q langchain-openai==0.1.7
!{sys.executable} -m pip install -q chromadb==0.4.15
!{sys.executable} -m pip install -q tiktoken
!{sys.executable} -m pip install -q beautifulsoup4
!{sys.executable} -m pip install -q requests
!{sys.executable} -m pip install -q lxml

print("\n✅ All packages installed successfully!")
print("📋 Next: Run Cell 2 to set your API key")

📦 Installing packages...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have num

## 🔑  Set API Key


In [2]:
import os

# ⚠️ PASTE YOUR OPENAI API KEY HERE
os.environ['OPENAI_API_KEY'] = "paste-your-API-key-here"

# Verify
if "paste-your-key" in os.environ.get('OPENAI_API_KEY', ''):
    print("❌ ERROR: Replace 'sk-proj-paste-your-key-here' with your real key!")
    print("\nGet it from: https://platform.openai.com/api-keys")
else:
    key = os.environ['OPENAI_API_KEY']
    print("✅ API key configured!")
    print(f"Starts with: {key[:20]}...")
    print("\n📋 Next: Run Cell 3 to download documents")

✅ API key configured!
Starts with: sk-proj--0oy4nYznIcb...

📋 Next: Run Cell 3 to download documents


## 📥Download SBA Documents

Downloads official SBA content.

In [3]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import time

def download_sba_documents():
    """Download SBA documents"""

    base_dir = Path('sba_documents')
    base_dir.mkdir(exist_ok=True)

    # SBA URLs
    urls = {
        'starting_business': [
            'https://www.sba.gov/business-guide/10-steps-start-your-business',
            'https://www.sba.gov/business-guide/plan-your-business/choose-business-structure',
            'https://www.sba.gov/business-guide/plan-your-business/write-your-business-plan',
        ],
        'government_contracting': [
            'https://www.sba.gov/federal-contracting/contracting-guide',
            'https://www.sba.gov/federal-contracting/contracting-guide/small-business-set-asides',
            'https://www.sba.gov/federal-contracting/contracting-assistance-programs/8a-business-development-program',
        ],
        'financing': [
            'https://www.sba.gov/funding-programs/loans',
            'https://www.sba.gov/funding-programs/loans/7a-loans',
            'https://www.sba.gov/funding-programs/loans/504-loans',
        ],
        'certifications': [
            'https://www.sba.gov/business-guide/grow-your-business/woman-owned-businesses',
            'https://www.sba.gov/business-guide/grow-your-business/veteran-owned-businesses',
            'https://www.sba.gov/partners/contracting-officials/small-business-procurement/hubzone-program',
        ]
    }

    print("📥 Downloading SBA documents...\n")

    total_downloaded = 0

    for category, url_list in urls.items():
        category_dir = base_dir / category
        category_dir.mkdir(exist_ok=True)

        for url in url_list:
            try:
                response = requests.get(url, timeout=15, headers={
                    'User-Agent': 'Mozilla/5.0'
                })

                soup = BeautifulSoup(response.text, 'html.parser')

                # Remove scripts and styles
                for element in soup(["script", "style", "nav", "footer", "header"]):
                    element.decompose()

                # Get text
                content = soup.get_text(separator='\n', strip=True)

                # Clean up
                lines = [line.strip() for line in content.split('\n') if line.strip()]
                content = '\n'.join(lines)

                # Save
                filename = url.split('/')[-1][:50] + '.txt'
                filepath = category_dir / filename

                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(content)

                total_downloaded += 1
                print(f"✅ [{total_downloaded}] {category}/{filename}")

                time.sleep(0.5)  # Be nice to SBA servers

            except Exception as e:
                print(f"⚠️  Failed: {url.split('/')[-1]} ({str(e)[:50]})")

    print(f"\n✅ Downloaded {total_downloaded} documents")
    return total_downloaded

# Download
doc_count = download_sba_documents()

if doc_count > 0:
    print("\n📋 Next: Run Cell 4 to build the RAG system")
else:
    print("\n❌ No documents downloaded - check your internet connection")

📥 Downloading SBA documents...

✅ [1] starting_business/10-steps-start-your-business.txt
✅ [2] starting_business/choose-business-structure.txt
✅ [3] starting_business/write-your-business-plan.txt
✅ [4] government_contracting/contracting-guide.txt
✅ [5] government_contracting/small-business-set-asides.txt
✅ [6] government_contracting/8a-business-development-program.txt
✅ [7] financing/loans.txt
✅ [8] financing/7a-loans.txt
✅ [9] financing/504-loans.txt
✅ [10] certifications/woman-owned-businesses.txt
✅ [11] certifications/veteran-owned-businesses.txt
✅ [12] certifications/hubzone-program.txt

✅ Downloaded 12 documents

📋 Next: Run Cell 4 to build the RAG system


## 🤖 Build RAG System

Creates the complete RAG system.

In [4]:
from pathlib import Path
from typing import List, Dict

# Import with fallbacks
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

print("\n" + "="*80)
print("BUILDING SBA RAG SYSTEM")
print("="*80)

# Step 1: Load documents
print("\n📚 Step 1: Loading documents...")
documents = []
docs_dir = Path('sba_documents')

for txt_file in docs_dir.rglob('*.txt'):
    with open(txt_file, 'r', encoding='utf-8') as f:
        content = f.read()

    doc = Document(
        page_content=content,
        metadata={
            'source': str(txt_file),
            'category': txt_file.parent.name,
            'filename': txt_file.name
        }
    )
    documents.append(doc)

print(f"✅ Loaded {len(documents)} documents")

# Step 2: Split into chunks
print("\n✂️  Step 2: Splitting into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks")

# Step 3: Create embeddings & vector store
print("\n🔢 Step 3: Creating vector database with OpenAI...")
print("   (This costs ~$0.01)")

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# Use in-memory vectorstore (no persistence issues!)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="sba_docs"
)

print("✅ Vector database created")

# Step 4: Create QA chain
print("\n🔗 Step 4: Setting up QA chain...")

prompt_template = """You are an expert assistant for small business owners using ONLY official SBA documents.

CRITICAL INSTRUCTIONS:
- You can ONLY use information from the context below
- If the context does not contain the answer, say "I don't have information about that in the SBA documents"
- Do NOT use your general knowledge
- Do NOT make up information
- ONLY answer if the context explicitly contains the information

Context from SBA documents:
{context}

Question: {question}

Answer (using ONLY the context above):"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

llm = OpenAI(temperature=0, model_name="gpt-3.5-turbo-instruct")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("✅ QA chain ready")

print("\n" + "="*80)
print("✅ RAG SYSTEM READY!")
print("="*80)
print("\n📋 Next: Run Cell 5 to test with questions")

# Make qa_chain available globally
globals()['qa_chain'] = qa_chain


BUILDING SBA RAG SYSTEM

📚 Step 1: Loading documents...
✅ Loaded 12 documents

✂️  Step 2: Splitting into chunks...
✅ Created 60 chunks

🔢 Step 3: Creating vector database with OpenAI...
   (This costs ~$0.01)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector database created

🔗 Step 4: Setting up QA chain...
✅ QA chain ready

✅ RAG SYSTEM READY!

📋 Next: Run Cell 5 to test with questions


## 🧪  Test with Sample Questions

Ask questions and see answers with sources!

In [5]:
def ask_question(question: str):
    """Ask a question and display results"""
    result = qa_chain({"query": question})

    print(f"\n💬 ANSWER:\n{result['result']}")

    print(f"\n📚 SOURCES:")
    for i, doc in enumerate(result['source_documents'], 1):
        category = doc.metadata.get('category', 'Unknown')
        filename = doc.metadata.get('filename', 'Unknown')
        print(f"   {i}. {category}/{filename}")

# Test questions
print("\n" + "="*80)
print("📝 TESTING THE SYSTEM")
print("="*80)

test_questions = [
    "What's the difference between an LLC and S-Corp?",
    "How do I get government contracts?",
    "What is the 8(a) program?",
    "What SBA loans are available?"
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'─'*80}")
    print(f"Q{i}: {q}")
    print('─'*80)
    ask_question(q)

print("\n" + "="*80)
print("✅ ALL TESTS PASSED!")
print("="*80)
print("\n📋 Next: Use Cell 6 to ask your own questions")


📝 TESTING THE SYSTEM

────────────────────────────────────────────────────────────────────────────────
Q1: What's the difference between an LLC and S-Corp?
────────────────────────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



💬 ANSWER:

I don't have information about that in the SBA documents.

📚 SOURCES:
   1. starting_business/write-your-business-plan.txt
   2. starting_business/10-steps-start-your-business.txt
   3. starting_business/10-steps-start-your-business.txt

────────────────────────────────────────────────────────────────────────────────
Q2: How do I get government contracts?
────────────────────────────────────────────────────────────────────────────────

💬 ANSWER:
 You can increase your chance of winning a government contract by researching the federal marketplace and taking advantage of SBA resources. Additionally, you can evaluate your small business to see if it has what it takes to win a government contract and make sure it meets the basic requirements for federal contracting. You can also learn about the different types of government contracts and how SBA determines business sizes. Finally, you can comply with regulations and responsibilities for government contracting and consider becom

## 💬 Ask Your Own Questions



In [6]:
# CHANGE THIS TO YOUR QUESTION
my_question = "Do I need a business license to start a business?"

print(f"\n❓ YOUR QUESTION:")
print(f"Q: {my_question}")
print("\n" + "─"*80)

ask_question(my_question)

print("\n" + "─"*80)
print("\n💡 TIP: Edit 'my_question' above and re-run this cell!")


❓ YOUR QUESTION:
Q: Do I need a business license to start a business?

────────────────────────────────────────────────────────────────────────────────

💬 ANSWER:
 Yes, you need a business license to start a business. According to the SBA documents, "Keep your business running smoothly by staying legally compliant. The licenses and permits you need for your business will vary by industry, state, location, and other factors." Additionally, the document "Learn more about licenses and permits" provides more information on obtaining the necessary licenses and permits for your business.

📚 SOURCES:
   1. starting_business/10-steps-start-your-business.txt
   2. starting_business/10-steps-start-your-business.txt
   3. starting_business/10-steps-start-your-business.txt

────────────────────────────────────────────────────────────────────────────────

💡 TIP: Edit 'my_question' above and re-run this cell!



- Complete RAG system with official SBA documents
- Semantic search using OpenAI embeddings
- Question answering with source citations
- No file management needed - all self-contained!



### 🚀 Key Features:
- ✅ Self-contained (no uploads)
- ✅ Fast responses (2-3 sec)
- ✅ Accurate with citations

---

## 📊 System Architecture:

```
User Question
     ↓
Vector Search (finds relevant chunks)
     ↓
Retrieved Context (3 most relevant chunks)
     ↓
OpenAI GPT-3.5 (generates answer)
     ↓
Answer + Source Citations
```

---


---

### 💻 SBA RAG simple UI

In [9]:
#  Gradio UI - Web Interface!

# Install Gradio
!pip install -q gradio

# Create UI
import gradio as gr

def answer_question(question):
    """Answer a question using the RAG system"""
    if not question.strip():
        return "Please enter a question!", ""

    try:
        # Use the qa_chain from Cell 4
        result = qa_chain({"query": question})

        # Format answer
        answer = result['result']

        # Format sources
        sources = []
        for doc in result['source_documents']:
            category = doc.metadata.get('category', 'Unknown')
            filename = doc.metadata.get('filename', 'Unknown')
            sources.append(f"• {category}/{filename}")

        sources_text = "\n".join(sources[:3])  # Top 3 sources

        return answer, sources_text

    except Exception as e:
        return f"Error: {str(e)}", ""

# Sample questions for easy testing
examples = [
    ["What's the difference between LLC and S-Corp?"],
    ["How do I get government contracts?"],
    ["What is the 8(a) program?"],
    ["What SBA loans are available?"],
    ["Do I need a business license?"]
]

# Create beautiful interface
demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(
        label="Your Question",
        placeholder="Ask anything about starting a business...",
        lines=3
    ),
    outputs=[
        gr.Textbox(label="💬 Answer", lines=10),
        gr.Textbox(label="📚 Sources", lines=5)
    ],
    title="🏢 SBA Business Assistant",
    description="""
    Ask me anything about starting and running a small business!
    I use official SBA documents to provide accurate answers with source citations.
    """,
    examples=examples,
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

# Launch with public URL!
print("\n🚀 Launching UI...")
print("⏳ This takes ~30 seconds...")
demo.launch(share=True)  # Creates shareable public URL!


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(



🚀 Launching UI...
⏳ This takes ~30 seconds...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d4b40c477794859e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🤖 SBA RAG + AGENT (Business advisor)


In [13]:
"""
============================================================
SBA RAG + AGENT -
============================================================
"""

!pip install langchain-core --quiet

print("✅ Packages installed")

# ============================================================
# STEP 2: Define Tools
# ============================================================

def search_sba_docs(query: str) -> str:
    """
    Search SBA documentation using existing RAG chain

    Args:
        query: Question about SBA programs, loans, certifications

    Returns:
        Answer from SBA documents
    """
    try:
        result = qa_chain({"query": query})
        return result['result']
    except Exception as e:
        return f"Error searching SBA docs: {str(e)}"


def calculate_loan_payment(principal: float, annual_rate: float, years: int) -> str:
    """
    Calculate monthly loan payment and totals

    Args:
        principal: Loan amount in dollars (e.g., 100000 for $100,000)
        annual_rate: Annual interest rate as percentage (e.g., 8 for 8%)
        years: Loan term in years (e.g., 10 for 10 years)

    Returns:
        Formatted string with payment details
    """
    try:
        # Convert annual rate to monthly decimal
        monthly_rate = (annual_rate / 100) / 12
        months = years * 12

        # Calculate monthly payment using amortization formula
        if monthly_rate == 0:
            monthly_payment = principal / months
        else:
            monthly_payment = principal * (
                monthly_rate * (1 + monthly_rate)**months
            ) / ((1 + monthly_rate)**months - 1)

        total_paid = monthly_payment * months
        total_interest = total_paid - principal

        # Format response
        response = f"""
Loan Calculation Results:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Principal Amount:    ${principal:,.2f}
Interest Rate:       {annual_rate}% annually
Loan Term:          {years} years ({months} months)

Monthly Payment:     ${monthly_payment:,.2f}
Total Amount Paid:   ${total_paid:,.2f}
Total Interest Paid: ${total_interest:,.2f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
        return response

    except Exception as e:
        return f"Error calculating loan: {str(e)}"


# Test the tools
print("🧪 Testing tools...")
print("\n1. Testing search_sba_docs:")
test_result = search_sba_docs("What is the 8(a) program?")
print(test_result[:200] + "...")

print("\n2. Testing calculate_loan_payment:")
test_calc = calculate_loan_payment(100000, 8, 10)
print(test_calc)

print("\n✅ Both tools working!")

# ============================================================
# STEP 3: Create LangChain Tools
# ============================================================

from langchain.tools import Tool
from langchain_openai import ChatOpenAI

# Wrap functions as LangChain tools
tools = [
    Tool(
        name="search_sba_documentation",
        func=search_sba_docs,
        description="""
        Use this tool to search SBA (Small Business Administration) documentation.
        Good for questions about:
        - SBA loan programs (7(a), 504, microloans, disaster loans)
        - Business certifications (8(a), WOSB, HUBZone, VOSB)
        - Eligibility requirements
        - Application processes
        - Program benefits and restrictions

        Input should be a clear question or search query.
        """
    ),
    Tool(
        name="calculate_loan_payment",
        func=lambda x: calculate_loan_payment(*[float(i.strip()) for i in x.split(',')]),
        description="""
        Use this tool to calculate monthly loan payments.

        Input format: "principal,annual_rate,years"
        Example: "100000,8,10" for $100k at 8% for 10 years

        Use when user asks about:
        - Monthly payments
        - Total interest
        - Loan amortization
        - Payment schedules
        """
    )
]

print(f"✅ Created {len(tools)} tools for agent")

# ============================================================
# STEP 4: Create Agent
# ============================================================

from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create agent prompt
system_message = """You are a helpful SBA (Small Business Administration) advisor assistant.

You have access to two tools:
1. search_sba_documentation - Search official SBA documents
2. calculate_loan_payment - Calculate loan payments and totals

When answering questions:
- Use search_sba_documentation for SBA policies, programs, and requirements
- Use calculate_loan_payment when user asks about loan payments or calculations
- For questions needing both (e.g., "What's the payment on an SBA 7(a) loan?"), use BOTH tools
- Always be clear, specific, and helpful
- You only answer business related questions
- If you don't know, say so honestly

Be professional but friendly. Your goal is to help small business owners understand SBA resources.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create agent
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Shows thinking process
    max_iterations=3,
    handle_parsing_errors=True
)

print("✅ Agent created successfully!")

# ============================================================
# STEP 5: Test the Agent
# ============================================================

print("\n" + "="*70)
print("🧪 TESTING AGENT")
print("="*70)

# Test 1: Simple SBA question
print("\nTest 1: SBA Question")
print("-" * 70)
response1 = agent_executor.invoke({"input": "What is the 8(a) program?"})
print("\nFinal Answer:")
print(response1['output'])

# Test 2: Calculation question
print("\n" + "="*70)
print("\nTest 2: Calculation")
print("-" * 70)
response2 = agent_executor.invoke({
    "input": "What's the monthly payment on a $100,000 loan at 8% interest for 10 years?"
})
print("\nFinal Answer:")
print(response2['output'])

# Test 3: Combined question (uses both tools!)
print("\n" + "="*70)
print("\nTest 3: Combined Question")
print("-" * 70)
response3 = agent_executor.invoke({
    "input": "I want to borrow $250,000 through SBA 7(a) at 9% for 20 years. What's my monthly payment and what are the requirements?"
})
print("\nFinal Answer:")
print(response3['output'])

print("\n" + "="*70)
print("✅ ALL TESTS COMPLETE!")
print("="*70)

# ============================================================
# STEP 6: Create Simple Function for Gradio
# ============================================================

def ask_agent(question: str) -> tuple:
    """
    Ask the SBA agent a question

    Returns:
        (answer, thinking_process)
    """
    try:
        # Run agent
        result = agent_executor.invoke({"input": question})

        answer = result['output']

        # Extract thinking process (from verbose output)
        # Note: This is simplified - in production you'd capture this differently
        thinking = "Agent used available tools to answer your question."

        return answer, thinking

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        return error_msg, "Agent encountered an error"

print("\n✅ Agent function ready for Gradio!")


print("\n" + "="*70)
print("🎯 AGENT Is READY!")
print("="*70)
print("""
The agent can now:
✅ Search SBA documentation
✅ Calculate loan payments
✅ Combine both for complete answers
""")


✅ Packages installed
🧪 Testing tools...

1. Testing search_sba_docs:
 The 8(a) program is a program offered by the Small Business Administration (SBA) that provides unique and valuable business assistance to experienced socially and economically disadvantaged small bus...

2. Testing calculate_loan_payment:

Loan Calculation Results:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Principal Amount:    $100,000.00
Interest Rate:       8% annually
Loan Term:          10 years (120 months)

Monthly Payment:     $1,213.28
Total Amount Paid:   $145,593.11
Total Interest Paid: $45,593.11
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


✅ Both tools working!
✅ Created 2 tools for agent
✅ Agent created successfully!

🧪 TESTING AGENT

Test 1: SBA Question
----------------------------------------------------------------------


> Entering new AgentExecutor chain...

Invoking: `search_sba_documentation` with `8(a) program`


 The 8(a) program is a business development program established by the U.S. Small Business Ad

## 🤖 SBA Business advisor UI

In [11]:
"""
============================================================
SBA RAG GRADIO UI WITH AGENT
============================================================
"""

import gradio as gr

def answer_with_agent(question):
    """
    Answer questions using AI agent (can search docs AND calculate)
    """

    try:
        # Use agent instead of basic RAG
        result = agent_executor.invoke({"input": question})
        answer = result['output']

        # Check if it's a "don't know" response
        no_answer_phrases = [
            "don't have information",
            "not in the sba documents",
            "cannot find",
            "don't know"
        ]

        is_no_answer = any(phrase in answer.lower() for phrase in no_answer_phrases)

        # Format answer
        formatted_answer = f"**Answer:**\n\n{answer}"

        # Determine sources
        if is_no_answer:
            formatted_sources = "🤷 No relevant sources found"
        else:
            # For agent responses, show which tools were used
            formatted_sources = "**Agent Tools Used:**\n\n"

            # Simple detection of which tools were used (based on answer content)
            tools_used = []

            if "$" in answer or "payment" in answer.lower() or "interest" in answer.lower():
                tools_used.append("💰 Loan Calculator")

            if "sba" in answer.lower() or "program" in answer.lower():
                tools_used.append("📚 SBA Documentation Search")

            if tools_used:
                for tool in tools_used:
                    formatted_sources += f"- {tool}\n"
            else:
                formatted_sources += "- 📚 SBA Documentation Search\n"

        return formatted_answer, formatted_sources

    except Exception as e:
        error_answer = f"**Error:**\n\n{str(e)}"
        error_sources = "❌ Agent encountered an error"
        return error_answer, error_sources


# ============================================================
# CREATE GRADIO INTERFACE WITH AGENT
# ============================================================

with gr.Blocks(title="SBA Business Advisor with AI Agent", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🤖 SBA Business Advisor (RAG + AI Agent)

    **Enhanced with AI Agent!** Now I can:
    - 📚 Search SBA documentation(RAG)
    - 💰 Calculate loan payments
    - 🔗 Combine both for complete answers

    """)

    with gr.Row():
        with gr.Column(scale=1):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g., What's the payment on a $100,000 SBA 7(a) loan at 8% for 10 years?",
                lines=4
            )

            with gr.Row():
                submit_btn = gr.Button("🚀 Ask Agent", variant="primary", scale=2)
                clear_btn = gr.ClearButton(components=[question_input], scale=1)

        with gr.Column(scale=1):
            answer_output = gr.Markdown(
                label="Answer",
                value="*Your answer will appear here...*"
            )

            sources_output = gr.Markdown(
                label="Tools & Sources",
                value="*Agent tools used will be shown here...*"
            )

    # Example questions showcasing agent capabilities
    gr.Markdown("### 💡 Example Questions")

    gr.Examples(
        examples=[
            # Basic SBA questions (uses search tool)
            ["What is the 8(a) Business Development program?"],
            ["How do I apply for an SBA loan?"],

            # Calculation questions (uses calculator tool)
            ["What's the monthly payment on a $100,000 loan at 8% for 10 years?"],
            ["Calculate payment for $250,000 at 9% for 20 years"],

            # Combined questions (uses BOTH tools!)
            ["I want a $200,000 SBA 7(a) loan at 8.5% for 15 years. What's my payment and what are the requirements?"],
            ["What's the monthly cost of borrowing $150,000 for 12 years at 7%, and what SBA loan programs could I use?"]
        ],
        inputs=[question_input],
        label=None
    )

    gr.Markdown("""
    ---
    **🤖 Agent Features:**
    - Automatically chooses the right tool(s) for your question
    - Can chain multiple tools together for complex questions
    - Shows which tools were used to answer
    """)

    # Connect button to agent function
    submit_btn.click(
        fn=answer_with_agent,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )

# Launch
demo.launch(share=True, debug=False)

print("✅ SBA RAG Agent UI launched!")
print("🤖 Agent is ready with 2 tools:")
print("   1. 📚 SBA Documentation Search")
print("   2. 💰 Loan Payment Calculator")
print("\n🔗 Share URL will be displayed above")

/tmp/ipykernel_1160/1339066653.py:66: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SBA Business Advisor with AI Agent", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://45f41fd784416c5c35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ SBA RAG Agent UI launched!
🤖 Agent is ready with 2 tools:
   1. 📚 SBA Documentation Search
   2. 💰 Loan Payment Calculator

🔗 Share URL will be displayed above
